In [1]:
import os
os.environ.pop("DATABRICKS_CLUSTER_ID", None)
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql("USE CATALOG sokoni_retail")
base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

In [2]:
from pyspark.sql import functions as F

inv = spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/inventory/"
)

print("rows:", inv.count())
print("distinct store-product-week:",
      inv.select("store_id", "product_id", "week_start").distinct().count())
print("distinct weeks:", inv.select("week_start").distinct().count())

inv.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in inv.columns
]).show(vertical=True)

inv.select(
    F.min(F.col("quantity_on_hand").cast("int")).alias("min_qty"),
    F.max(F.col("quantity_on_hand").cast("int")).alias("max_qty"),
    (F.avg(F.when(F.col("quantity_on_hand") == 0, 1).otherwise(0)) * 100).alias("zero_stock_pct"),
).show()

rows: 1056000
distinct store-product-week: 1056000
distinct weeks: 96
-RECORD 0---------------
 store_id         | 0   
 product_id       | 0   
 week_start       | 0   
 quantity_on_hand | 0   

+-------+-------+------------------+
|min_qty|max_qty|    zero_stock_pct|
+-------+-------+------------------+
|      0|     49|18.360984848484847|
+-------+-------+------------------+



In [3]:
spark.read.option("header", True).csv(
    "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/inventory/"
).show(10, truncate=False)

+--------+----------+----------+----------------+
|store_id|product_id|week_start|quantity_on_hand|
+--------+----------+----------+----------------+
|S01     |P0126     |2025-01-06|18              |
|S02     |P0127     |2025-01-13|18              |
|S03     |P0128     |2025-01-20|33              |
|S04     |P0129     |2025-01-27|0               |
|S05     |P0130     |2025-02-03|6               |
|S06     |P0131     |2025-02-10|44              |
|S07     |P0132     |2025-02-17|0               |
|S08     |P0133     |2025-02-24|2               |
|S09     |P0134     |2025-03-03|0               |
|S10     |P0135     |2025-03-10|32              |
+--------+----------+----------+----------------+
only showing top 10 rows


In [2]:
f = spark.table("sokoni_retail.gold.fact_inventory_weekly")
st = spark.table("sokoni_retail.gold.dim_store")
p = spark.table("sokoni_retail.gold.dim_product")
d = spark.table("sokoni_retail.gold.dim_date")

print("no store match:", f.join(st, "store_id", "left_anti").count())
print("no product match:", f.join(p, "product_id", "left_anti").count())
print("no date match:", f.join(d, "date_key", "left_anti").count())

no store match: 0
no product match: 0
no date match: 11000


In [3]:
f = spark.table("sokoni_retail.gold.fact_inventory_weekly")
d = spark.table("sokoni_retail.gold.dim_date")

print("no date match:", f.join(d, "date_key", "left_anti").count())

no date match: 0
